# Study Double Zernike (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-12  
**Last Modified:** 2026-04-12  
**Status:** Draft  
**Keywords:** Double Zernike, AOS, intrinsics, correlations, LUT, rotator, elevation, thermal  

## Description

Aggregate Double Zernike (DZ) fit results across all pipeline chunks (OCS or CCS
coordinate system) and run cross-coefficient and operational analyses.

Key functionality:
1. Load and concatenate all `*_fits.parquet` (OCS) or `*_ccs_fits.parquet` (CCS)
   from the pipeline `output/` tree.
2. Reproduce DZ-vs-thermal scatter plots from `intrinsics_thermal_correlations`.
3. Full pairwise correlation analysis over all DZ coefficients; pull out the
   10 largest off-diagonal correlations and make scatter plots.
4. LUT exploration: DZ terms vs rotator angle (elevation ∈ 70 ± 3 deg) and
   DZ terms vs elevation (rotator angle ∈ 0 ± 3 deg).

**Output:** PDF in `output/dz_analysis_{coord_sys}.pdf`  
**Based on:** intrinsics_thermal_correlations.ipynb, run_pipeline outputs

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-12 | Aaron Roodman | Initial version |
| 2026-04-14 | Aaron Roodman | Add thermal-by-variable grid; split large plot sets into separate PDFs; add k1-k2 pairwise scan by (j1,j2); unify radial-trio color scale |
| 2026-04-19 | Aaron Roodman | Extend thermal grid and pairwise scan to j=4..17; add DZ × thermal correlation-matrix heatmap (§6.2) |
| 2026-05-11 | Aaron Roodman | Rename to study_doublezernike; split fits_input_dir from output_root (output now under output/study_doublezernike/); new `binning` parameter (bin_2x / bin_1x / None). |
| 2026-05-11 | Aaron Roodman | Consolidate all thermal plots into a single `<label>_thermal.pdf` (renamed from `_thermal_by_variable.pdf`).  Per-variable grid now covers every DZ term in the fit, split into two pages per thermal variable at `thermal_grid_j_split = 16`.  Drop radial DZ correlation (§12.1) and radial trio (§12.2) sections — no longer used. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Thermal Correlations — per-DZ-term](#thermal-per-term)
6. [Thermal Correlations — overview](#thermal-by-var)
    - [6.1 Per-variable grid — all DZ terms](#thermal-by-var-grid)
    - [6.2 DZ × thermal correlation matrix](#thermal-by-var-matrix)
7. [DZ Correlation Matrix](#corr-matrix-sec)
8. [Large DZ Correlations Grouped by Pupil Pair](#corr-groups)
9. [Expected Astigmatism-Symmetry Correlations](#corr-expected)
10. [Pairwise (k1,j1) vs (k2,j2) scan](#pairwise)
11. [LUT: DZ vs Rotator and Elevation](#lut)
12. [Output PDFs](#output)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Coordinate system for fit tables: 'OCS' or 'CCS'
coord_sys = 'OCS'

# Where to look for input fit parquet files (relative to this
# notebook).  The output of this notebook lives separately in
# `output_root` below.
fits_input_dir = 'output'

# Where all of this notebook's outputs go.  A per-run subdirectory
# (`output_label`) is then nested under this root.
output_root = 'output/study_doublezernike'

# Glob patterns: OCS uses *_fits.parquet, CCS uses *_ccs_fits.parquet
fit_glob = {
    'OCS': '*[!_ccs]_fits.parquet',
    'CCS': '*_ccs_fits.parquet',
}[coord_sys]

# ----- Binning selector -----
# Convenience filter that maps directly onto param_set_filter:
#     binning = 'bin_2x'  ->  param_set_filter = 'bin_2x'
#     binning = 'bin_1x'  ->  param_set_filter = 'bin_1x'
#     binning = None      ->  no binning filter (use whatever
#                              chunk_names / param_set_filter say).
# Explicit chunk_names or param_set_filter take precedence if set.
binning = 'bin_2x'

# ----- Chunk selection -----
# Pick ONE mode (leave the others None):
#   chunk_names       — explicit list (or single string) of run names
#                       from runs.yaml, e.g.
#                         chunk_names = 'chunk5'
#                         chunk_names = ['chunk1', 'chunk2', 'chunk3']
#   param_set_filter  — single param_set name or substring, matches any
#                       chunk whose param_set contains it,
#                       e.g. 'fam_danish_v1_triplets'
#   (both None)       — load every *_fits.parquet found under output_root
chunk_names = None
param_set_filter = None

# DZ fit prefix to use for analysis (z1toz3 or z1toz6)
dz_prefix = 'z1toz6'

# Quality cuts
max_coeff_um = 2.0  # reject visits with any DZ coefficient > this (µm)

# Thermal variables (only used if present in the merged tables)
thermal_vars = [
    'cam_air_temp', 'm2_air_temp', 'm1m3_air_temp', 'outside_temp',
    'm2_delta_t', 'cam_m1m3_delta_t', 'dome_delta_t',
    'x_gradient', 'y_gradient', 'z_gradient', 'radial_gradient',
    'tma_truss_temp_pxpy', 'tma_truss_temp_mxmy',
]

# DZ terms to plot vs thermal (k = focal Noll, j = pupil Noll).
# Include all focal k for pupil j = 4 (defocus) plus astig/trefoil combos.
dz_terms_for_thermal = [
    (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4),  # all k, pupil defocus
    (5, 5), (6, 6),                                    # astig-astig
    (5, 10), (6, 9),                                   # astig-trefoil
]

# Per-variable thermal grid: focal k range covered on each page,
# and the split point that separates low-j from high-j pupil
# Zernikes.  Each thermal variable gets TWO pages: j < split on
# the first, j >= split on the second.  The pupil-j list itself
# is taken from `nollIndices` so every Z in the fit gets shown.
thermal_grid_k_range = range(1, 7)   # k = 1..6 (6 panels per col)
thermal_grid_j_split = 16            # j < 16  on page 1
                                     # j >= 16 on page 2

# Pairwise (k1,j1) vs (k2,j2) scan: one page per (j1, j2) pair
# j extended to 17 so 2nd Coma Zernike terms are captured.
pairwise_j_range = [j for j in range(4, 27) if j not in (20, 21)]
                                        # 21 standard Zj (Z4..Z19, Z22..Z26) -> 441 pages
pairwise_k_range = range(1, 7)          # k = 1..6 (36 subplots per page)

# Correlation scatter plots: minimum |r| to include
corr_threshold = 0.6

# Expected astigmatism-symmetry correlations in the (k, j) basis.
expected_astig_pairs = [
    ((5, 5), (6, 6)),
    ((6, 5), (5, 6)),
    ((1, 5), (1, 6)),
    ((4, 5), (4, 6)),
    ((2, 5), (3, 6)),
    ((3, 5), (2, 6)),
]

# LUT slice tolerances (all in degrees)
elev_center_for_rotator_scan = 70.0
elev_tol = 3.0
rot_center_for_elev_scan = 0.0
rot_tol = 3.0

# Per-run output subdirectory under output_root.  If None, a label is
# auto-generated after QC from coord_sys, the chunk filter, and the
# loaded day_obs range.  Set explicitly to override, e.g.
#     output_label = 'dz_analysis_OCS_danish_v1_mar-apr'
output_label = None

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def find_fit_files(output_root, coord_sys,
                   chunk_names=None, param_set_filter=None):
    """Locate DZ fit parquet files for the requested coord system.

    Selection modes (at most one should be set):
      * chunk_names — list of run names from runs.yaml; each is resolved
        via run_pipeline.fits_path / fits_path_ccs.
      * param_set_filter — string; any chunk whose param_set *contains*
        this string is selected.
      * None — glob output_root for every matching fit file (default).

    OCS files end '_fits.parquet' (and NOT '_ccs_fits.parquet').
    CCS files end '_ccs_fits.parquet'.
    """
    is_ccs = coord_sys == 'CCS'

    def _only_coord(paths):
        if is_ccs:
            return [p for p in paths if str(p).endswith('_ccs_fits.parquet')]
        return [p for p in paths
                if str(p).endswith('_fits.parquet')
                and not str(p).endswith('_ccs_fits.parquet')]

    # Accept a bare string for chunk_names as a convenience
    if isinstance(chunk_names, str):
        chunk_names = [chunk_names]

    if chunk_names or param_set_filter:
        # Resolve specific chunks from runs.yaml
        sys.path.insert(0, 'code')
        from run_pipeline import (
            load_runs as _load_runs,
            load_param_sets as _load_param_sets,
            resolve_run as _resolve_run,
            fits_path as _fits_path,
            fits_path_ccs as _fits_path_ccs,
        )
        data = _load_runs()
        param_sets = _load_param_sets()
        runs = data.get('runs', {})

        if chunk_names:
            selected = [(n, runs[n]) for n in chunk_names if n in runs]
            missing = [n for n in chunk_names if n not in runs]
            if missing:
                raise ValueError(f'Unknown chunk(s) in runs.yaml: {missing}')
        else:
            selected = [(n, cfg) for n, cfg in runs.items()
                        if param_set_filter in (cfg.get('param_set') or '')]
            if not selected:
                raise ValueError(
                    f'No runs in runs.yaml match param_set_filter='
                    f'{param_set_filter!r}')

        paths = []
        for name, cfg in selected:
            resolved = _resolve_run(cfg, param_sets)
            fp = _fits_path_ccs(resolved) if is_ccs else _fits_path(resolved)
            # run_pipeline returns 'output/...'; treat as repo-relative
            fp_path = Path(fp)
            if not fp_path.is_absolute():
                fp_path = Path(output_root).parent / fp_path if \
                    Path(output_root).name != 'output' else fp_path
            # Normal case: run_pipeline's fits_path already uses output/...
            paths.append(Path(fp))
        paths = _only_coord(paths)
        # Only return files that actually exist
        return [p for p in paths if Path(p).exists()]

    # Fall-through: glob every fit file in output_root
    root = Path(output_root)
    all_fits = sorted(root.glob('*_fits.parquet'))
    return _only_coord(all_fits)


def load_all_fits(files, source_label='source'):
    """Load and concatenate fit parquet tables, tagging each row with its source."""
    frames = []
    for p in files:
        df = pd.read_parquet(p)
        df[source_label] = Path(p).stem
        frames.append(df)
        print(f"  {Path(p).name}: {len(df)} rows, {len(df.columns)} cols")
    if not frames:
        raise FileNotFoundError('No fit parquet files found')
    combined = pd.concat(frames, ignore_index=True, sort=False)
    return combined


def dz_coeff_columns(df, prefix):
    """Return all DZ coefficient columns for a given prefix.

    Columns are named '{prefix}_z{j}_c{k}' where j = pupil Noll, k = focal Noll.
    Excludes '_err', '_scale', '_bad_fit'.
    """
    pattern = re.compile(rf'^{re.escape(prefix)}_z\d+_c\d+$')
    return [c for c in df.columns if pattern.match(c)]


def parse_jk(col, prefix):
    """Parse '(j, k)' from a column like 'z1toz6_z9_c5'."""
    m = re.match(rf'^{re.escape(prefix)}_z(\d+)_c(\d+)$', col)
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))


# Noll-index to human-readable name mappings
from common.zernike_names import (
    NOLL_NAMES, NOLL_FORMULAS, FOCAL_NAMES, PUPIL_NAMES,
)
def dz_label(col, prefix):
    """Descriptive label like 'Focus of Coma_x (k=4, j=7)'."""
    jk = parse_jk(col, prefix)
    if jk is None:
        return col
    j, k = jk
    focal = FOCAL_NAMES.get(k, f'k{k}')
    pupil = PUPIL_NAMES.get(j, f'Z{j}')
    return f'{focal} of {pupil} (k={k}, j={j})'


def short_label(col, prefix):
    j, k = parse_jk(col, prefix)
    return f'(k={k}, j={j})'


def quality_cut(df, prefix, max_coeff_um):
    """Apply standard quality cuts: drop bad_fit and outliers."""
    n0 = len(df)
    bad_col = f'{prefix}_bad_fit'
    if bad_col in df.columns:
        df = df[~df[bad_col].astype(bool)].copy()
    elif 'bad_fit' in df.columns:
        df = df[~df['bad_fit'].astype(bool)].copy()
    cols = dz_coeff_columns(df, prefix)
    out_mask = df[cols].abs().gt(max_coeff_um).any(axis=1)
    df = df[~out_mask].copy()
    print(f'Quality cut: {len(df)}/{n0} rows kept '
          f'(prefix={prefix}, |c| < {max_coeff_um} \u00b5m)')
    return df


def selection_tag(chunk_names, param_set_filter):
    """Short slug describing the chunk selection, for the output label.

    Accepts chunk_names as either a list/tuple or a single string
    (the latter is promoted to a one-element list, mirroring the
    convenience handling in find_fit_files).

    Returns '' when both filters are None (i.e. use everything found).
    """
    if chunk_names:
        if isinstance(chunk_names, str):
            chunk_names = [chunk_names]
        return '-'.join(chunk_names)
    if param_set_filter:
        # Normalize to a filesystem-safe slug
        return re.sub(r'[^A-Za-z0-9_]+', '_', param_set_filter).strip('_')
    return ''


<a id='data'></a>
## Data Access

In [ ]:
# Resolve binning -> param_set_filter when no explicit chunk_names
# or param_set_filter has been supplied.
if (chunk_names is None and param_set_filter is None
        and binning is not None):
    param_set_filter = binning
    print(f'binning={binning!r} -> param_set_filter={param_set_filter!r}')

fit_files = find_fit_files(
    fits_input_dir, coord_sys,
    chunk_names=chunk_names, param_set_filter=param_set_filter)
print(f'Found {len(fit_files)} {coord_sys} fit files in '
      f'{fits_input_dir}/')
if chunk_names:
    print(f'  chunk_names filter: {chunk_names}')
elif param_set_filter:
    print(f'  param_set_filter: {param_set_filter!r}')
for p in fit_files:
    print(f'  {p}')

df_all = load_all_fits(fit_files, source_label='chunk')
print(f'\nCombined: {len(df_all)} rows x {len(df_all.columns)} columns')
print(f'day_obs range: {df_all["day_obs"].min()} - {df_all["day_obs"].max()}')

In [ ]:
df = quality_cut(df_all, dz_prefix, max_coeff_um)
dz_cols = dz_coeff_columns(df, dz_prefix)
print(f'Number of DZ coefficient columns: {len(dz_cols)}')

# Sorted, unique pupil-Noll-j values represented in dz_cols.
# Used by the thermal grid (j_lo / j_hi split) and any other
# downstream cell that needs the j set actually fit.
iZs = sorted({parse_jk(c, dz_prefix)[0] for c in dz_cols
              if parse_jk(c, dz_prefix) is not None})
print(f'Pupil Noll indices in fit ({len(iZs)}): {iZs}')

# Unit normalization:
#   * alt is stored in RADIANS (from Butler meta) -> add alt_deg in degrees
#   * rotator_angle is already in DEGREES
if 'alt' in df.columns:
    df['alt_deg'] = np.rad2deg(df['alt'].astype(float))
    print(f'alt range (rad): {df["alt"].min():.3f} .. {df["alt"].max():.3f}')
    print(f'alt range (deg): {df["alt_deg"].min():.2f} .. {df["alt_deg"].max():.2f}')
if 'rotator_angle' in df.columns:
    print(f'rotator_angle range (deg): '
          f'{df["rotator_angle"].min():.2f} .. {df["rotator_angle"].max():.2f}')

# Sanity check on key metadata columns
for col in ['rotator_angle', 'alt', 'alt_deg', 'az'] + thermal_vars:
    n_valid = df[col].notna().sum() if col in df.columns else 0
    flag = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col:<25s} {flag:<8s} valid={n_valid}/{len(df)}')
# ----- Resolve per-run output subdirectory now that data is loaded -----
_dmin = int(df['day_obs'].min())
_dmax = int(df['day_obs'].max())
if output_label is None:
    _sel = selection_tag(chunk_names, param_set_filter)
    if _sel:
        output_label = f'dz_analysis_{coord_sys}_{_sel}_{_dmin}_{_dmax}'
    else:
        output_label = f'dz_analysis_{coord_sys}_{_dmin}_{_dmax}'
output_dir = Path(output_root) / output_label
output_dir.mkdir(parents=True, exist_ok=True)

output_pdf_main          = str(output_dir / f'{output_label}.pdf')
output_pdf_thermal       = str(output_dir / f'{output_label}_thermal.pdf')
# Intermediate PDFs the thermal sections stream into.  save-pdf merges
# them into output_pdf_thermal at the end.
output_pdf_thermal_scatter_tmp = str(output_dir / f'{output_label}_thermal_scatter_tmp.pdf')
output_pdf_thermal_grid_tmp    = str(output_dir / f'{output_label}_thermal_grid_tmp.pdf')
output_pdf_corr_groups   = str(output_dir / f'{output_label}_correlation_groups.pdf')
output_pdf_pairwise      = str(output_dir / f'{output_label}_pairwise.pdf')

print(f'\nOutput directory: {output_dir}')
for p in [output_pdf_main, output_pdf_thermal,
          output_pdf_corr_groups, output_pdf_pairwise]:
    print(f'  {p}')

<a id='thermal-per-term'></a>
## 5. Thermal Correlations — per-DZ-term

For each selected DZ coefficient `(k, j)` produce a page of scatter plots
vs each thermal variable. Each point is one visit; a linear fit and Pearson
`r` are overlaid. Pages for this section are appended to the main PDF.

In [ ]:
def thermal_scatter_page(df, dz_col, k, j, thermal_vars):
    """Make a page of scatter plots: dz_col vs every thermal variable."""
    have = [tv for tv in thermal_vars if tv in df.columns]
    n = len(have)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2 * nrows))
    fig.suptitle(
        f'DZ (k={k}, j={j}) = {dz_col}  vs thermal variables',
        fontsize=13, y=1.01)
    axes = np.atleast_1d(axes).ravel()
    for idx, tv in enumerate(have):
        ax = axes[idx]
        m = df[dz_col].notna() & df[tv].notna()
        x = df.loc[m, tv].values
        y = df.loc[m, dz_col].values
        ax.scatter(x, y, s=10, alpha=0.6, edgecolors='none')
        if len(x) > 2:
            c = np.polyfit(x, y, 1)
            r = np.corrcoef(x, y)[0, 1]
            xf = np.linspace(x.min(), x.max(), 50)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
            ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes,
                    fontsize=9, va='top',
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))
        ax.set_xlabel(tv, fontsize=8)
        ax.set_ylabel(f'(k={k},j={j}) [µm]', fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    return fig

In [ ]:
# Stream per-(k, j) thermal scatter pages straight into the
# intermediate PDF; plt.close() after each to bound peak memory.
# The pages are merged into output_pdf_thermal by save-pdf.
Path(output_pdf_thermal_scatter_tmp).parent.mkdir(parents=True, exist_ok=True)
_thermal_scatter_pages = 0
with PdfPages(output_pdf_thermal_scatter_tmp) as _pdf:
    for k, j in dz_terms_for_thermal:
        col = f'{dz_prefix}_z{j}_c{k}'
        if col not in df.columns:
            print(f'Skipping {col}: not in table')
            continue
        if not any(tv in df.columns for tv in thermal_vars):
            print('No thermal variables present in tables — skipping section.')
            break
        fig = thermal_scatter_page(df, col, k, j, thermal_vars)
        _pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)
        _thermal_scatter_pages += 1
print(f'Wrote {_thermal_scatter_pages} thermal scatter page(s) -> '
      f'{output_pdf_thermal_scatter_tmp}')


<a id='thermal-by-var'></a>
## 6. Thermal Correlations — overview

Two views of DZ–thermal correlations, written together to
`{output_pdf_thermal}`:

- **6.1 Per-variable grid** — for each thermal variable a page with a
  6×14 grid of DZ scatter plots (focal `k = 1..6`, pupil `j = 4..17`).
- **6.2 DZ × thermal correlation matrix** — single heatmap of Pearson `r`
  between every DZ coefficient and every thermal variable.

<a id='thermal-by-var-grid'></a>
### 6.1 Per-variable grid — all DZ terms

For each thermal variable, **two pages**: the first covers focal `k = 1..6`
× pupil `j < thermal_grid_j_split` (default `j < 16`), the second covers
the remaining pupil j values from `nollIndices` (`j >= 16`).  Every DZ
coefficient in the fit (typically 6 × 21 = 126 cells across the two
pages) gets a scatter against the thermal variable.

In [ ]:
# Per-variable thermal correlation grids — build TWO pages per
# variable (one for j < thermal_grid_j_split, one for the rest of the
# pupil-j list from nollIndices).  Figures are accumulated in
# `thermal_grid_figs`; the §13 save-pdf cell streams them into the
# consolidated thermal PDF.
def thermal_variable_grid_page(df, tv, k_range, j_range, dz_prefix,
                               page_tag=''):
    """One page: grid of DZ(k,j) vs single thermal variable tv."""
    k_list = list(k_range)
    j_list = list(j_range)
    nrows = len(k_list)
    ncols = len(j_list)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(1.6 * ncols, 1.5 * nrows + 0.8),
        sharex=True,
        layout='constrained')
    axes = np.atleast_2d(axes)
    fig.suptitle(
        f'DZ coefficients vs {tv}  ({coord_sys}, {dz_prefix})'
        + (f'  —  {page_tag}' if page_tag else ''),
        fontsize=13)

    for ri, k in enumerate(k_list):
        for ci, j in enumerate(j_list):
            ax = axes[ri, ci]
            col = f'{dz_prefix}_z{j}_c{k}'
            if col not in df.columns:
                ax.set_visible(False)
                continue
            m = df[col].notna() & df[tv].notna()
            x = df.loc[m, tv].values
            y = df.loc[m, col].values
            if len(x) < 3:
                ax.set_visible(False)
                continue
            ax.scatter(x, y, s=4, alpha=0.5, edgecolors='none')
            c = np.polyfit(x, y, 1)
            xf = np.linspace(x.min(), x.max(), 30)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.0, alpha=0.8)
            r = np.corrcoef(x, y)[0, 1]
            ax.text(0.04, 0.93, f'r={r:+.2f}', transform=ax.transAxes,
                    fontsize=6, va='top',
                    bbox=dict(boxstyle='round,pad=0.1',
                              fc='white', alpha=0.7, lw=0))
            ax.axhline(0, color='gray', lw=0.3, alpha=0.5)
            ax.tick_params(labelsize=6)
            if ri == 0:
                ax.set_title(f'j={j} ({PUPIL_NAMES.get(j, f"Z{j}")})',
                             fontsize=7)
            if ci == 0:
                ax.set_ylabel(f'k={k}\n{FOCAL_NAMES.get(k, f"k{k}")}',
                              fontsize=7)
            if ri == nrows - 1:
                ax.set_xlabel(tv, fontsize=7)
    return fig


# Pupil-j split: lower half on page 1, upper half on page 2.
j_lo = [j for j in iZs if j <  thermal_grid_j_split]
j_hi = [j for j in iZs if j >= thermal_grid_j_split]
print(f'thermal grid pages: '
      f'page 1 j = {j_lo} ({len(j_lo)} terms);  '
      f'page 2 j = {j_hi} ({len(j_hi)} terms)')

# Stream pages directly into the intermediate PDF; close each fig
# immediately so peak memory is bounded by ~1 figure at a time.
Path(output_pdf_thermal_grid_tmp).parent.mkdir(parents=True, exist_ok=True)
_thermal_grid_pages = 0
with PdfPages(output_pdf_thermal_grid_tmp) as _pdf:
    for tv in thermal_vars:
        if tv not in df.columns:
            print(f'Skipping {tv}: not in table')
            continue
        if j_lo:
            fig = thermal_variable_grid_page(
                df, tv, thermal_grid_k_range, j_lo, dz_prefix,
                page_tag=f'page 1/2 (j={j_lo[0]}..{j_lo[-1]})')
            _pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            _thermal_grid_pages += 1
        if j_hi:
            fig = thermal_variable_grid_page(
                df, tv, thermal_grid_k_range, j_hi, dz_prefix,
                page_tag=f'page 2/2 (j={j_hi[0]}..{j_hi[-1]})')
            _pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            _thermal_grid_pages += 1
print(f'Wrote {_thermal_grid_pages} thermal-grid page(s) -> '
      f'{output_pdf_thermal_grid_tmp}')

<a id='thermal-by-var-matrix'></a>
### 6.2 DZ × thermal correlation matrix

Single Pearson-r heatmap of every DZ coefficient × every thermal
variable.  Closes out the thermal PDF.

In [ ]:
# DZ x thermal Pearson correlation matrix
# Rows: DZ coefficients (ordered by j, then k), Cols: thermal variables.
k_list = list(thermal_grid_k_range)
# Use the full set of pupil-j present in the fit (iZs from qc-cell)
j_list = list(iZs)
thermal_available = [tv for tv in thermal_vars if tv in df.columns]

dz_keys = [(j, k) for j in j_list for k in k_list]
dz_cols_grid = [f'{dz_prefix}_z{j}_c{k}' for (j, k) in dz_keys]
# Keep only columns actually in df
dz_cols_grid = [c for c in dz_cols_grid if c in df.columns]
dz_keys = [parse_jk(c, dz_prefix) for c in dz_cols_grid]

corr_mat = np.full((len(dz_cols_grid), len(thermal_available)), np.nan)
for i, col in enumerate(dz_cols_grid):
    for c_idx, tv in enumerate(thermal_available):
        m = df[col].notna() & df[tv].notna()
        if m.sum() > 2:
            corr_mat[i, c_idx] = np.corrcoef(
                df.loc[m, col].values, df.loc[m, tv].values)[0, 1]

# Row labels: "(k=K, j=J) Focal of Pupil"
row_labels = []
for (j, k) in dz_keys:
    fname = FOCAL_NAMES.get(k, f'k{k}')
    pname = PUPIL_NAMES.get(j, f'Z{j}')
    row_labels.append(f'k={k} j={j}  {fname} of {pname}')

# Separator indices between pupil-j groups (rows)
sep_rows = []
for i in range(1, len(dz_keys)):
    if dz_keys[i][0] != dz_keys[i - 1][0]:
        sep_rows.append(i - 0.5)

fig_dz_thermal_corr, ax = plt.subplots(
    figsize=(max(7, 0.5 * len(thermal_available) + 4),
             max(10, 0.18 * len(dz_cols_grid) + 1.2)),
    layout='constrained')
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(thermal_available)))
ax.set_xticklabels(thermal_available, rotation=70, fontsize=7,
                   ha='right')
ax.set_yticks(range(len(dz_cols_grid)))
ax.set_yticklabels(row_labels, fontsize=6)
for y in sep_rows:
    ax.axhline(y, color='black', lw=0.4, alpha=0.5)
ax.set_title(
    f'DZ × thermal Pearson correlation ({dz_prefix}, {coord_sys})\n'
    f'{len(dz_cols_grid)} DZ coefficients × {len(thermal_available)} '
    f'thermal variables',
    fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.6, label='r')
plt.show()

# Quick print: top |r| per thermal variable
print('Top |r| per thermal variable:')
for c_idx, tv in enumerate(thermal_available):
    col = corr_mat[:, c_idx]
    if np.all(np.isnan(col)):
        continue
    i_best = int(np.nanargmax(np.abs(col)))
    print(f'  {tv:<25s}  r={col[i_best]:+.3f}  {row_labels[i_best]}')

<a id='corr-matrix-sec'></a>
## 7. DZ Correlation Matrix

Compute the pairwise Pearson correlation matrix over all DZ coefficients
and display it as a heatmap.

In [ ]:
corr_df = df[dz_cols].dropna()
print(f'Correlation sample size: {len(corr_df)} visits, {len(dz_cols)} DZ terms')
corr_matrix = corr_df.corr(method='pearson')

fig_corr, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = [dz_label(c, dz_prefix) for c in dz_cols]
ax.set_xticks(range(len(dz_cols)))
ax.set_xticklabels(labels, rotation=90, fontsize=5)
ax.set_yticks(range(len(dz_cols)))
ax.set_yticklabels(labels, fontsize=5)
ax.set_title(f'Pearson correlation — all {dz_prefix} DZ coefficients ({coord_sys})')
plt.colorbar(im, ax=ax, shrink=0.8, label='r')
plt.show()

<a id='corr-groups'></a>
## 8. Large DZ Correlations Grouped by Pupil Pair

Extract pairs with `|r| > corr_threshold`, group by pupil-Zernike pair
`(j_lo, j_hi)`, and display up to 8 scatter plots per page. Within each
group the plots are ordered by `(k_a, k_b)`, and the lower pupil Noll
index is always on the x-axis. All pages go to
`{output_pdf_corr_groups}`.

In [ ]:
# Extract all pairs with |r| > corr_threshold, grouped by pupil-Zernike pair
from collections import defaultdict

corr_threshold = 0.6

n = len(dz_cols)
iu, ju = np.triu_indices(n, k=1)
rs = corr_matrix.values[iu, ju]

# Filter by threshold
sig_mask = np.abs(rs) > corr_threshold
sig_indices = np.where(sig_mask)[0]
print(f'Pairs with |r| > {corr_threshold}: {len(sig_indices)} '
      f'(out of {len(rs)} total)')

# Group by sorted pupil-Zernike pair (j_lo, j_hi)
pupil_groups = defaultdict(list)
for idx in sig_indices:
    a = dz_cols[iu[idx]]
    b = dz_cols[ju[idx]]
    r = rs[idx]
    ja, ka = parse_jk(a, dz_prefix)
    jb, kb = parse_jk(b, dz_prefix)
    # Ensure col_a has the lower pupil index
    if ja > jb:
        a, b = b, a
        ja, ka, jb, kb = jb, kb, ja, ka
    pupil_groups[(ja, jb)].append((a, b, ka, kb, r))

# Sort within each group by (ka, kb)
for key in pupil_groups:
    pupil_groups[key].sort(key=lambda t: (t[2], t[3]))

# Print summary
print(f'\n{len(pupil_groups)} pupil-Zernike pair groups:')
for (ja, jb) in sorted(pupil_groups.keys()):
    pa = PUPIL_NAMES.get(ja, f'Z{ja}')
    pb = PUPIL_NAMES.get(jb, f'Z{jb}')
    entries = pupil_groups[(ja, jb)]
    print(f'  {pa} (Z{ja}) × {pb} (Z{jb}): {len(entries)} pairs')
    for a, b, ka, kb, r in entries:
        print(f'    {dz_label(a, dz_prefix)}  vs  {dz_label(b, dz_prefix)}  r={r:+.3f}')

In [ ]:
# Grouped scatter plots: one set of pages per pupil-Zernike pair.
# Stream pages directly to the correlation-groups PDF and close each
# figure so they don't accumulate in memory.
Path(output_pdf_corr_groups).parent.mkdir(parents=True, exist_ok=True)
plots_per_page = 8
ncols_per_page = 4
n_corr_pages = 0

with PdfPages(output_pdf_corr_groups) as pdf:
    for (ja, jb) in sorted(pupil_groups.keys()):
        entries = pupil_groups[(ja, jb)]
        pa = PUPIL_NAMES.get(ja, f'Z{ja}')
        pb = PUPIL_NAMES.get(jb, f'Z{jb}')
        if ja == jb:
            page_title = f'{pa} (Z{ja}) self-correlations ({coord_sys})'
        else:
            page_title = (f'{pa} (Z{ja}) × {pb} (Z{jb}) '
                          f'correlations ({coord_sys})')

        for page_start in range(0, len(entries), plots_per_page):
            page_entries = entries[page_start:page_start + plots_per_page]
            nrows_page = int(np.ceil(len(page_entries) / ncols_per_page))
            fig, axes = plt.subplots(
                nrows_page, ncols_per_page,
                figsize=(4.5 * ncols_per_page, 4.2 * nrows_page + 0.9),
                layout=None)
            axes = np.atleast_1d(axes).ravel()

            for idx, (a, b, ka, kb, r) in enumerate(page_entries):
                ax = axes[idx]
                m = df[a].notna() & df[b].notna()
                x = df.loc[m, a].values
                y = df.loc[m, b].values
                ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
                if len(x) > 2:
                    c = np.polyfit(x, y, 1)
                    xf = np.linspace(x.min(), x.max(), 50)
                    ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
                ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
                ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
                ax.set_xlabel(dz_label(a, dz_prefix), fontsize=8)
                ax.set_ylabel(dz_label(b, dz_prefix), fontsize=8)
                ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=10)

            for idx in range(len(page_entries), len(axes)):
                axes[idx].set_visible(False)

            page_num = page_start // plots_per_page + 1
            total_pages = int(np.ceil(len(entries) / plots_per_page))
            suffix = f' ({page_num}/{total_pages})' if total_pages > 1 else ''
            fig.suptitle(page_title + suffix, fontsize=13)
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            plt.show()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_corr_pages += 1

print(f'\nWrote {n_corr_pages} correlation-group pages to '
      f'{output_pdf_corr_groups}')
corr_page_figs = []   # legacy compatibility

<a id='corr-expected'></a>
### 9. Expected Astigmatism-Symmetry Correlations

Scatter plots of DZ coefficient pairs expected to correlate by the
astigmatism symmetry between focal Noll k = 5 ↔ 6 and pupil Noll
j = 5 ↔ 6 (`expected_astig_pairs`). This page is appended to the main PDF.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(expected_astig_pairs) / ncols))
fig_expected, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.0 * nrows))
axes = np.atleast_1d(axes).ravel()

for idx, ((ka, ja), (kb, jb)) in enumerate(expected_astig_pairs):
    a = f'{dz_prefix}_z{ja}_c{ka}'
    b = f'{dz_prefix}_z{jb}_c{kb}'
    ax = axes[idx]
    if a not in df.columns or b not in df.columns:
        ax.set_visible(False)
        print(f'Skipping ({ka},{ja}) vs ({kb},{jb}): column missing')
        continue
    m = df[a].notna() & df[b].notna()
    x = df.loc[m, a].values
    y = df.loc[m, b].values
    r = np.corrcoef(x, y)[0, 1] if len(x) > 2 else np.nan
    ax.scatter(x, y, s=12, alpha=0.6, edgecolors='none')
    if len(x) > 2:
        c = np.polyfit(x, y, 1)
        xf = np.linspace(x.min(), x.max(), 50)
        ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.2, alpha=0.8)
    ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax.set_xlabel(f'(k={ka}, j={ja})  [µm]', fontsize=10)
    ax.set_ylabel(f'(k={kb}, j={jb})  [µm]', fontsize=10)
    ax.set_title(f'r = {r:+.3f}   (n={m.sum()})', fontsize=11)

for idx in range(len(expected_astig_pairs), len(axes)):
    axes[idx].set_visible(False)

fig_expected.suptitle(
    f'Expected astigmatism-symmetry DZ correlations ({dz_prefix}, {coord_sys})',
    fontsize=13, y=1.005)
plt.show()

<a id='pairwise'></a>
## 10. Pairwise (k1,j1) vs (k2,j2) scan

For every combination of pupil Noll indices `j1, j2 = 4..14` produce a
page with 36 scatter plots covering the 6×6 grid of focal Noll indices
`k1, k2 = 1..6`. Each subplot shows `(k2,j2)` (y-axis) vs `(k1,j1)`
(x-axis), titled `(k2,j2) vs (k1,j1)`. Total: 11 × 11 = 121 pages,
written to `{output_pdf_pairwise}`.

In [ ]:
# Pairwise (k1,j1) vs (k2,j2) scan — one page per (j1, j2) combination.
# Each page has a 6x6 grid for k1, k2 = 1..6.
# With 121 pages, we write directly to the PDF here rather than keeping
# all figures in memory.
def pairwise_page(df, j1, j2, k_range, dz_prefix):
    k_list = list(k_range)
    n = len(k_list)
    fig, axes = plt.subplots(
        n, n,
        figsize=(2.0 * n, 2.0 * n + 0.8),
        layout='constrained')
    axes = np.atleast_2d(axes)
    p1 = PUPIL_NAMES.get(j1, f'Z{j1}')
    p2 = PUPIL_NAMES.get(j2, f'Z{j2}')
    fig.suptitle(
        f'({p2} Z{j2}) vs ({p1} Z{j1})  —  j2={j2} vs j1={j1}  ({coord_sys})',
        fontsize=12)

    for ri, k2 in enumerate(k_list):
        for ci, k1 in enumerate(k_list):
            ax = axes[ri, ci]
            col_x = f'{dz_prefix}_z{j1}_c{k1}'
            col_y = f'{dz_prefix}_z{j2}_c{k2}'
            if col_x not in df.columns or col_y not in df.columns:
                ax.set_visible(False)
                continue
            # Skip identical-column self-correlation when (k1,j1)==(k2,j2)
            if col_x == col_y:
                ax.set_visible(False)
                continue
            m = df[col_x].notna() & df[col_y].notna()
            if m.sum() < 3:
                ax.set_visible(False)
                continue
            x = df.loc[m, col_x].values
            y = df.loc[m, col_y].values
            ax.scatter(x, y, s=4, alpha=0.5, edgecolors='none')
            c = np.polyfit(x, y, 1)
            xf = np.linspace(x.min(), x.max(), 30)
            ax.plot(xf, np.polyval(c, xf), 'r-', lw=1.0, alpha=0.8)
            r = np.corrcoef(x, y)[0, 1]
            ax.axhline(0, color='gray', lw=0.3, alpha=0.5)
            ax.axvline(0, color='gray', lw=0.3, alpha=0.5)
            ax.set_title(
                f'(k={k2},j={j2}) vs (k={k1},j={j1})  r={r:+.2f}',
                fontsize=7)
            ax.tick_params(labelsize=6)
            if ri == n - 1:
                ax.set_xlabel(f'k1={k1} ({FOCAL_NAMES.get(k1,"")})',
                              fontsize=7)
            if ci == 0:
                ax.set_ylabel(f'k2={k2} ({FOCAL_NAMES.get(k2,"")})',
                              fontsize=7)
    return fig


# Write all pages directly to the pairwise PDF
Path(output_pdf_pairwise).parent.mkdir(parents=True, exist_ok=True)
j_list = list(pairwise_j_range)
n_total = len(j_list) ** 2
print(f'Writing {n_total} pairwise pages to {output_pdf_pairwise}...')

with PdfPages(output_pdf_pairwise) as pdf:
    page_ct = 0
    for j1 in j_list:
        for j2 in j_list:
            fig = pairwise_page(df, j1, j2, pairwise_k_range, dz_prefix)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            page_ct += 1
            if page_ct % 20 == 0:
                print(f'  ...wrote {page_ct}/{n_total}')
print(f'Done: {page_ct} pages -> {output_pdf_pairwise}')

<a id='lut'></a>
## 11. LUT: DZ vs Rotator and Elevation

Inputs to a future Look-Up Table for AOS feed-forward correction.

* **Rotator scan** — plot every DZ coefficient vs `rotator_angle` for visits
  with `alt` within ±3 deg of 70 deg.
* **Elevation scan** — plot every DZ coefficient vs `alt` for visits with
  `rotator_angle` within ±3 deg of 0 deg.

In [ ]:
def grid_scan_page(df, x_col, dz_cols, prefix, title, x_label, x_unit='deg'):
    """Plot every DZ coefficient vs x_col in a tight grid."""
    n = len(dz_cols)
    ncols = 6
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(2.3 * ncols, 1.9 * nrows),
                             sharex=True)
    axes = np.atleast_1d(axes).ravel()
    for idx, col in enumerate(dz_cols):
        ax = axes[idx]
        m = df[col].notna() & df[x_col].notna()
        x = df.loc[m, x_col].values
        y = df.loc[m, col].values
        ax.scatter(x, y, s=6, alpha=0.6, edgecolors='none')
        ax.axhline(0, color='gray', lw=0.5, alpha=0.6)
        ax.set_title(short_label(col, prefix) if prefix in col
                     else col, fontsize=7)
        ax.tick_params(labelsize=6)
    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)
    fig.suptitle(title, fontsize=12, y=1.005)
    fig.supxlabel(f'{x_label} [{x_unit}]', fontsize=10)
    fig.supylabel('DZ coefficient [µm]', fontsize=10)
    return fig

In [ ]:
# DZ vs rotator angle, at fixed elevation band (all in degrees)
elev_lo = elev_center_for_rotator_scan - elev_tol
elev_hi = elev_center_for_rotator_scan + elev_tol
mask_elev = df['alt_deg'].between(elev_lo, elev_hi)
df_rot = df[mask_elev & df['rotator_angle'].notna()].copy()
print(f'Elevation slice {elev_lo:.0f}–{elev_hi:.0f} deg: '
      f'{len(df_rot)}/{len(df)} visits')

if len(df_rot) >= 5:
    fig_rot = grid_scan_page(
        df_rot, 'rotator_angle', dz_cols, dz_prefix,
        title=(f'DZ vs rotator angle  (elevation = '
               f'{elev_center_for_rotator_scan:.0f} ± {elev_tol:.0f} deg, '
               f'n={len(df_rot)}, {coord_sys})'),
        x_label='rotator angle')
    plt.show()
else:
    fig_rot = None
    print('Too few visits in elevation slice for plot.')

In [ ]:
# DZ vs elevation (degrees), at fixed rotator-angle band
rot_lo = rot_center_for_elev_scan - rot_tol
rot_hi = rot_center_for_elev_scan + rot_tol
mask_rot = df['rotator_angle'].between(rot_lo, rot_hi)
df_elev = df[mask_rot & df['alt_deg'].notna()].copy()
print(f'Rotator slice {rot_lo:.0f}–{rot_hi:.0f} deg: '
      f'{len(df_elev)}/{len(df)} visits')

if len(df_elev) >= 5:
    fig_elev = grid_scan_page(
        df_elev, 'alt_deg', dz_cols, dz_prefix,
        title=(f'DZ vs elevation  (rotator = '
               f'{rot_center_for_elev_scan:.0f} ± {rot_tol:.0f} deg, '
               f'n={len(df_elev)}, {coord_sys})'),
        x_label='elevation')
    plt.show()
else:
    fig_elev = None
    print('Too few visits in rotator slice for plot.')

<a id='output'></a>
## 12. Output PDFs

The analysis produces three PDFs in
`{output_root}/{output_label}/`:

* `{output_label}.pdf` — main PDF (full-DZ correlation matrix,
  expected astigmatism-symmetry pairs, LUT scans).
* `{output_label}_thermal.pdf` — every thermal-related plot:
  per-DZ-term scatter pages (§5), per-variable 6 × n_j grids
  (§6.1, two pages per thermal variable), and the
  DZ × thermal correlation matrix (§6.2).
* `{output_label}_correlation_groups.pdf` — grouped-by-pupil-pair
  scatter pages for `|r| > corr_threshold` (§8).
* `{output_label}_pairwise.pdf` — pairwise (k1,j1) × (k2,j2) scan
  pages (§10).

In [ ]:
# Main PDF — the small, single-figure sections.  Thermal content
# lives entirely in output_pdf_thermal (assembled below); radial
# sections were dropped.
Path(output_pdf_main).parent.mkdir(parents=True, exist_ok=True)
with PdfPages(output_pdf_main) as pdf:
    pdf.savefig(fig_corr, bbox_inches='tight')
    plt.close(fig_corr)
    pdf.savefig(fig_expected, bbox_inches='tight')
    plt.close(fig_expected)
    if fig_rot is not None:
        pdf.savefig(fig_rot, bbox_inches='tight')
        plt.close(fig_rot)
    if fig_elev is not None:
        pdf.savefig(fig_elev, bbox_inches='tight')
        plt.close(fig_elev)
print(f'Wrote main PDF: {output_pdf_main}')

# Thermal PDF: merge the two intermediate streamed PDFs and the
# DZ × thermal correlation matrix into the final consolidated file.
# Uses pypdf (preferred) or PyPDF2 (fallback) for the merge.
_thermal_parts = []
for _p in (output_pdf_thermal_scatter_tmp, output_pdf_thermal_grid_tmp):
    if Path(_p).exists():
        _thermal_parts.append(_p)

# Render the corr-matrix figure to a one-page intermediate buffer.
import io as _io
_corr_buf = None
if 'fig_dz_thermal_corr' in dir() and fig_dz_thermal_corr is not None:
    _corr_buf = _io.BytesIO()
    with PdfPages(_corr_buf) as _pdf:
        _pdf.savefig(fig_dz_thermal_corr, bbox_inches='tight')
    plt.close(fig_dz_thermal_corr)
    _corr_buf.seek(0)

_merged_ok = False
try:
    try:
        from pypdf import PdfWriter as _PdfWriter
    except ImportError:
        from PyPDF2 import PdfWriter as _PdfWriter
    _writer = _PdfWriter()
    for _p in _thermal_parts:
        _writer.append(_p)
    if _corr_buf is not None:
        _writer.append(_corr_buf)
    with open(output_pdf_thermal, 'wb') as _fh:
        _writer.write(_fh)
    _writer.close()
    _merged_ok = True
    # Tidy up the intermediates only after a successful merge.
    for _p in _thermal_parts:
        try:
            Path(_p).unlink()
        except OSError:
            pass
    print(f'Wrote thermal PDF (merged): {output_pdf_thermal}')
except Exception as _e:
    print(f'(pypdf merge failed: {type(_e).__name__}: {_e})')
    print('Thermal sections left as separate files:')
    for _p in _thermal_parts:
        print(f'  {_p}')
    if _corr_buf is not None:
        _fallback = str(Path(output_pdf_thermal).with_name(
            Path(output_pdf_thermal).stem + '_corr_matrix.pdf'))
        with open(_fallback, 'wb') as _fh:
            _fh.write(_corr_buf.getvalue())
        print(f'  {_fallback}')

# (Grouped |r|>threshold scatter pages -> own PDF, written streaming
# by §8; pairwise PDF likewise by §10 — nothing to do here.)
